In [36]:
from dataclasses import dataclass
 
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
from models.vit_gpt2_model import ViTGPT2FromScratch

In [3]:
m = ViTGPT2FromScratch()
px = torch.randn(2, 3, 224, 224)
ids = torch.randint(0, 50257, (2, 12))
out = m(px, ids, labels=ids.clone())
print("logits", tuple(out["logits"].shape), "loss", float(out["loss"]))
gen = m.generate(px, bos_id=50256, eos_id=50256, max_len=10, temperature=0.0)
print("generated", tuple(gen.shape))
print("params(M)", sum(p.numel() for p in m.parameters()) / 1e6)

/tmp/ipykernel_34501/3106291990.py:5: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:852.)
  print("logits", tuple(out["logits"].shape), "loss", float(out["loss"]))


logits (2, 12, 50257) loss 10.796475410461426
generated (2, 11)
params(M) 239.19744


In [4]:
from models.vit_gpt2_model import build_vit_gpt2_for_finetune, build_vit_gpt2_pretrained, caption

In [5]:
model_pretrained, img_processor_pretrained, tokenizer_pretrained = build_vit_gpt2_pretrained(device=device)

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from pathlib import Path
from PIL import Image

paths = sorted(Path('data/samples').glob("*.jpg"))
imgs = [Image.open(p).convert("RGB") for p in paths]

In [9]:
caption_pretrained = caption(model=model_pretrained, image_processor=img_processor_pretrained, tokenizer=tokenizer_pretrained, images=imgs, device=device)

In [12]:
import matplotlib.pyplot as plt
import textwrap

def compare_captioned(images, captions, cols=3, scale=4):
    n = len(images)
    cols = min(cols, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(scale * cols, scale * rows), squeeze=False)
    axes = axes.ravel()
    for ax, img, cap in zip(axes, images, captions):
        ax.imshow(img)
        ax.set_title(textwrap.fill(cap, 30), fontsize=10)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("big_grid.png", dpi=100, bbox_inches="tight")
    plt.close()

compare_captioned(images=imgs, captions=caption_pretrained)

___

# BLIP Transfer Learning

In [1]:
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import textwrap

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

device

'cuda'

In [6]:
from models.blip_based_model import BLIPCaptioner